# TOC -> OCR (tables) -> LLM #1 (pipeline réel, pas de données synthétiques)

Ce notebook chaîne, sur un **vrai PDF**, exactement dans le style de
`04_document_structure_analysis.ipynb` (mêmes classes, mêmes appels réels,
pas de reconstruction/synthèse) :

```text
PDF réel
  -> PageAnalyzer (src/document_structure/page_analysis.py)
  -> TOCDetector.detect_printed_tocs (src/document_structure/toc.py)      [TOC détecté]
  -> OCR table (src/ocr/ocr_pipeline.py) sur les pages du TOC             [re-extraction]
  -> fusion native + OCR -> contenu organisé par région de TOC
  -> classify_sections (src/llm/classifier.py) -- une fois par TOC détecté
```

Rien n'est réécrit : chaque étape appelle le code du projet tel quel.


## 0. Configuration -- même PDF que le notebook de référence


In [24]:
from pathlib import Path
import sys
import json as json_lib

import pandas as pd

import pymupdf as fitz

sys.path.insert(
    0,
    r"C:\Users\camelia\Downloads\last version 3"
    
)

PDF_PATH = r"C:\Users\camelia\Downloads\projets\projet1OCP\data\MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf"


print("PDF_PATH:", PDF_PATH)


PDF_PATH: C:\Users\camelia\Downloads\projets\projet1OCP\data\MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf


## 1. PageAnalyzer -- extraction native (texte + tables), inchangé


In [2]:
from src.document_structure.page_analysis import PageAnalyzer

doc = fitz.open(str(PDF_PATH))
page_analyzer = PageAnalyzer()
pages_meta = page_analyzer.extract_document(doc)

print("Pages analysées :", len(pages_meta))


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Pages analysées : 1068


## 2. TOCDetector -- détection réelle du/des Table of Contents

Exactement `detect_printed_tocs()`, comme dans le notebook de référence
(section 9). C'est la LISTE DE VRAIES `TOCRegion` qu'on va ensuite traiter
une par une.


In [3]:
from src.document_structure.toc import TOCDetector

toc_detector = TOCDetector()
detected_tocs = toc_detector.detect_printed_tocs(pages_meta)

print("TOC détectés :", len(detected_tocs))
for i, region in enumerate(detected_tocs, 1):
    print(f"  region {i}: pages={region.page_numbers} type={region.toc_type} "
          f"confidence={round(region.confidence, 3)} entries={len(region.entries)}")


TOC détectés : 5
  region 1: pages=[5, 6] type=printed confidence=0.5 entries=69
  region 2: pages=[188] type=printed confidence=0.5 entries=36
  region 3: pages=[409] type=printed confidence=0.71 entries=77
  region 4: pages=[635] type=printed confidence=0.5 entries=3
  region 5: pages=[794] type=printed confidence=0.621 entries=31


## 3. Transformer chaque page en image, puis appliquer le pipeline OCR

Le pipeline OCR du projet (`src/ocr/ocr_pipeline.py`) travaille sur une
**image** (tableau numpy), pas directement sur le PDF. On rend donc les
deux étapes explicites, dans l'ordre :

1. **page PDF -> image** : `load_pdf_page_as_image()` (`src/ocr/preprocessing.py`,
   déjà utilisé par `OCRPipeline.process()` en interne -- on l'appelle ici
   nous-mêmes pour voir l'image intermédiaire) rend la page en image OpenCV
   BGR via PyMuPDF (`get_pixmap`), au DPI du projet (`DEFAULT_DPI`).
2. **image -> pipeline OCR** : on enchaîne alors, sur CETTE image, les
   étapes du pipeline telles qu'exposées par `OCRPipeline`
   (`detect_tables` -> `recognize_table_structure` -> `ocr_cells` ->
   `reconstruct_table`) -- ce sont exactement les étapes internes de
   `OCRPipeline.process()`, simplement rendues explicites ici pour
   inspecter chaque résultat intermédiaire (table détectée, cellules,
   DataFrame reconstruit).

Le DataFrame obtenu est ensuite reparsé avec `TOCDetector._parse_table()`
-- le même parseur que pour une table native, pas de logique dupliquée.

Si `paddleocr` n'est pas installé (`OCRUnavailableError`) ou si l'OCR
échoue sur une page donnée, on le signale et on continue avec les seules
entrées natives pour cette page -- une page ne doit jamais faire échouer
tout le notebook (même principe de dégradation gracieuse que
`pipeline/page_processing.py`).


In [7]:
%pip cache purge

Files removed: 3880 (3775.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
%rmdir /s /q "%USERPROFILE%\.paddlex\official_models"

In [10]:
%pip install paddlepaddle==3.2.2 paddleocr==3.3.0

  Using cached paddleocr-3.3.0-py3-none-any.whl.metadata (34 kB)
Using cached paddleocr-3.3.0-py3-none-any.whl (81 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 8.4 MB/s eta 0:00:12
   - -------------------------------------- 3.1/101.7 MB 9.7 MB/s eta 0:00:11
   - -------------------------------------- 4.2/101.7 MB 9.3 MB/s eta 0:00:11
   - -------------------------------------- 4.2/101.7 MB 9.3 MB/s eta 0:00:11
   -- ------------------------------------- 5.2/101.7 MB 5.5 MB/s eta 0:00:18
   -- ------------------------------------- 6.3/101.7 MB 4.9 MB/s eta 0:00:20
   --- ------------------------------------ 7.9/101.7 MB 5.3 MB/s eta 0:00:18
   --- ------------------------------------ 9.4/101.7 MB 5.8 MB/s eta 0:00:16
   ---- ----------------------------------- 11.5/101.7 MB 6.2 MB/s eta 0:00:15
   ----- ---------------------------------- 13.4/101.7 MB 6.4 MB/s eta 0:00:14
   ----- ---------------------------------- 14.7/101.7 MB 6.6 MB/s eta 0:00:14
   ------ --------------------------------- 16.8/101.7 MB 6.8 MB/s e


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!{sys.executable} -m pip show paddleocr

Name: paddleocr
Version: 3.3.0
Summary: Awesome multilingual OCR and document parsing toolkits based on PaddlePaddle
Home-page: https://github.com/PaddlePaddle/PaddleOCR
Author: 
Author-email: PaddlePaddle <paddleocr@baidu.com>
License: Apache License 2.0
Location: c:\Users\camelia\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: paddlex, PyYAML, requests, typing-extensions
Required-by: 


In [4]:
import numpy as np

from src.document_structure.models import TableRepresentation
from src.ocr.preprocessing import load_pdf_page_as_image
from src.ocr.config import DEFAULT_DPI


def render_page_as_image(pdf_path, pdf_page_number: int, dpi: int = DEFAULT_DPI) -> np.ndarray:
    """Etape 1 : page PDF (1-indexee) -> image OpenCV BGR.
    Meme fonction que celle utilisee en interne par OCRPipeline.process()
    (src/ocr/preprocessing.py), appelee ici explicitement."""
    return load_pdf_page_as_image(str(pdf_path), page_number=pdf_page_number - 1, dpi=dpi)


def run_ocr_pipeline_on_image(ocr_pipeline, image: np.ndarray) -> pd.DataFrame:
    """Etape 2 : image -> pipeline OCR, en reproduisant explicitement les
    etapes internes de OCRPipeline.process() (detection de layout -> table
    dominante -> structure de table -> OCR par cellule -> reconstruction),
    pour pouvoir inspecter chaque etape intermediaire."""
    tables = ocr_pipeline.detect_tables(image)
    if not tables:
        return pd.DataFrame()

    table = tables[0]  # table dominante de la page
    coordinate = table.get("coordinate", table.get("bbox"))
    x1, y1, x2, y2 = map(int, coordinate)
    table_image = image[y1:y2, x1:x2]

    structure = ocr_pipeline.recognize_table_structure(table_image)

    # Les coordonnees des cellules sont relatives a table_image -> les
    # replacer dans le repere de l'image complete de la page.
    for cell in structure["cells"]:
        cell["bbox"][0] += x1
        cell["bbox"][1] += y1
        cell["bbox"][2] += x1
        cell["bbox"][3] += y1
        cell["cx"] += x1
        cell["cy"] += y1

    cells = ocr_pipeline.ocr_cells(image, structure["cells"])
    return ocr_pipeline.reconstruct_table(cells)


def ocr_table_entries_for_page(pdf_path, pdf_page_number: int):
    """Chaine complete : page -> image -> pipeline OCR -> DataFrame ->
    TOCEntry (via le meme parseur que les tables natives)."""
    try:
        from src.ocr.ocr_pipeline import  OCRPipeline
    except Exception as exc:
        print(f"  [page {pdf_page_number}] OCR indisponible (import) : {exc}")
        return []

    try:
        ocr_pipeline = OCRPipeline()
        image = render_page_as_image(pdf_path, pdf_page_number)
        print(f"  [page {pdf_page_number}] image rendue : {image.shape[1]}x{image.shape[0]}px")
        df = run_ocr_pipeline_on_image(ocr_pipeline, image)
    except Exception as exc:
        print(f"  [page {pdf_page_number}] OCR table a echoue : {exc}")
        return []

    if df is None or df.empty:
        print(f"  [page {pdf_page_number}] OCR : aucune table reconstruite")
        return []

    table = TableRepresentation(
        rows=[list(df.columns)] + df.values.tolist(),
        extraction_method="paddleocr",
    )
    return toc_detector._parse_table(table, source_page=pdf_page_number)


def ocr_text_for_page(pdf_path, pdf_page_number: int) -> str:
    """Meme fonction que pipeline/page_processing.py pour le texte OCR
    plein-page (Tesseract)."""
    try:
        from src.ocr.text_ocr import ocr_pdf_page_text
        return ocr_pdf_page_text(str(pdf_path), page_number=pdf_page_number - 1)
    except Exception as exc:
        print(f"  [page {pdf_page_number}] OCR texte a echoue : {exc}")
        return ""


# Appliquer l'OCR table sur toutes les pages de tous les TOC detectes.
ocr_entries_by_region = []
for i, region in enumerate(detected_tocs, 1):
    print(f"\n--- Region {i} (pages {region.page_numbers}) ---")
    region_ocr_entries = []
    for pdf_page in region.page_numbers:
        entries = ocr_table_entries_for_page(PDF_PATH, pdf_page)
        print(f"  page {pdf_page}: {len(entries)} entree(s) via OCR table")
        region_ocr_entries.extend(entries)
    ocr_entries_by_region.append(region_ocr_entries)



--- Region 1 (pages [5, 6]) ---


c:\Users\camelia\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
c:\Users\camelia\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\camelia\.paddlex\official_models\PP-DocLayout_plus-L`.
Model files already exist. Using cached files. To redownload, pl

  [page 5] image rendue : 2481x3509px


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\camelia\.paddlex\official_models\PP-DocLayout_plus-L`.


  page 5: 26 entree(s) via OCR table


...

  [page 635] image rendue : 2481x3509px


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\camelia\.paddlex\official_models\PP-DocLayout_plus-L`.


  page 635: 9 entree(s) via OCR table

--- Region 5 (pages [794]) ---


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\camelia\.paddlex\official_models\SLANet`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\camelia\.paddlex\official_models\PP-OCRv5_server_rec`.


  [page 794] image rendue : 2481x3509px
  [page 794] OCR : aucune table reconstruite
  page 794: 0 entree(s) via OCR table


## 4. Organiser : fusionner entrées natives + entrées OCR par région

Pour chaque `TOCRegion`, on combine `region.entries` (natif,
`TOCDetector`/PyMuPDF) et les entrées OCR équivalentes, en dédupliquant sur
le texte du titre -- on garde la version native quand elle existe (elle
vient d'un vrai texte/table PDF), et on ajoute les entrées supplémentaires
que seul l'OCR a trouvées.


In [22]:
def organize_region(region, ocr_entries):
    seen_titles = {e.text.strip().lower() for e in region.entries if e.text}
    merged = list(region.entries)
    added_from_ocr = 0
    for entry in ocr_entries:
        key = (entry.text or "").strip().lower()
        if key and key not in seen_titles:
            merged.append(entry)
            seen_titles.add(key)
            added_from_ocr += 1
    return merged, added_from_ocr


organized_regions = []
for i, (region, ocr_entries) in enumerate(zip(detected_tocs, ocr_entries_by_region), 1):
    merged, added = organize_region(region, ocr_entries)
    organized_regions.append(merged)
    print(f"Region {i}: {len(region.entries)} natives + {added} ajoutees par OCR = {len(merged)} au total")

# Inspection : la region la plus fournie
if organized_regions:
    biggest = max(range(len(organized_regions)), key=lambda i: len(organized_regions[i]))
    print(f"\nRegion {biggest + 1} (la plus fournie), premieres entrees organisees :")
    df_preview = pd.DataFrame([e.to_dict() for e in organized_regions[biggest][::]])
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    display(df_preview.head(100))


Region 1: 69 natives + 49 ajoutees par OCR = 118 au total
Region 2: 36 natives + 22 ajoutees par OCR = 58 au total
Region 3: 77 natives + 0 ajoutees par OCR = 77 au total
Region 4: 3 natives + 9 ajoutees par OCR = 12 au total
Region 5: 31 natives + 0 ajoutees par OCR = 31 au total

Region 1 (la plus fournie), premieres entrees organisees :


,text,section_number,level,printed_page_ref,reference_kind,source_page,confidence
0,DOCUMENTS&DRAWINGSLIST,1.1,2,P1-REF-LST-12-123-007,doc_code,5,0.8
1,SUBVENDORLIST,1.2,2,P1-REF-LST-12-123-003,doc_code,5,0.8
2,NOISEDATASHEET,1.3,2,P1-REF-DSS-12-123-008,doc_code,5,0.8
3,LUBRICANTLIST,1.4,2,P1-REF-SPC-12-123-002,doc_code,5,0.8
4,PAINTINGSPECIFICATIONS,1.5,2,P1-REF-SPC-12-123-003,doc_code,5,0.8
5,PACKAGEDRAWINGS,2,1,NaN,NaN,5,0.8
6,PROCESSFLOWDIAGRAM,2.1,2,P1-REF-2012-123-014,doc_code,5,0.8
7,P&IDREFRIGERATIONUNIT,2.2,2,P1-REF-2012-123-001,doc_code,5,0.8
8,GENERALARRANGEMENTDRAWING,2.3,2,P1-REF-2012-123-003,doc_code,5,0.8
9,FOUNDATIONLAYOUT,2.4,2,P1-REF-2012-123-012,doc_code,5,0.8


## 5. Passer chaque contenu de Table of Contents au LLM #1

Une région de TOC = un appel à `classify_sections()` (même prompt, même
schéma `SectionClassificationResult` que la production,
`src/llm/classifier.py` inchangé). Chaque entrée organisée de la région
devient un "section candidate" avec son titre, son numéro de section et
sa page imprimée -- c'est le contenu réel du sommaire, pas un extrait du
corps du document.


In [25]:
from src.llm.config import ENABLE_LLM, LLM_MODEL
import importlib
import src.llm.classifier
importlib.reload(src.llm.classifier)
importlib.reload(src.llm.prompts)
importlib.reload(src.llm.schemas)
from src.llm.classifier import classify_sections

def region_entries_to_payload(region_index: int, entries) -> list[dict]:
    payload = []
    for j, entry in enumerate(entries):
        payload.append({
            "section_id": f"toc{region_index}_e{j}",
            "title": entry.text,
            "section_number": entry.section_number,
            "page_start": entry.printed_page_ref,
            "page_end": entry.printed_page_ref,
            "level": entry.level,
            "confidence": round(entry.confidence, 3),
            "text": "",  
        })
    return payload


results_by_region = []

for i, entries in enumerate(organized_regions, 1):
    payload = region_entries_to_payload(i, entries)
    print(f"\n=== TOC region {i} --  {len(payload)} sections candidates ===")

    if not ENABLE_LLM:
        print("OPENAI_API_KEY absente -> appel non effectue. Payload pret :")
        print(json_lib.dumps(payload[:5], ensure_ascii=False, indent=2), "..." if len(payload) > 5 else "")
        results_by_region.append(None)
        continue

    try:
        result = classify_sections(payload)
    except Exception as exc:
        print(f"LLM #1 a echoue pour la region {i} : {exc}")
        results_by_region.append(None)
        continue

    print("Sections retenues :", result.selected_sections)
    display(pd.DataFrame([c.model_dump() for c in result.sections]))
    results_by_region.append(result)



=== TOC region 1 --  118 sections candidates ===
Sections retenues : ['toc1_e11', 'toc1_e12', 'toc1_e17', 'toc1_e19', 'toc1_e21', 'toc1_e23', 'toc1_e25', 'toc1_e27', 'toc1_e29', 'toc1_e39', 'toc1_e42', 'toc1_e46', 'toc1_e50', 'toc1_e52', 'toc1_e60', 'toc1_e100']


,section_id,relevance_score,reason,potential_information
0,toc1_e0,10,The title suggests a list of documents and dra...,[]
1,toc1_e1,10,A subvendor list is unlikely to provide direct...,[]
2,toc1_e2,20,Noise data sheets may contain some operational...,[]
3,toc1_e3,10,A lubricant list is unlikely to contain specif...,[]
4,toc1_e4,20,Painting specifications may include some detai...,[]
5,toc1_e5,20,Package drawings may provide some context but ...,[]
6,toc1_e6,20,Process flow diagrams may provide context for ...,[]
7,toc1_e7,30,P&ID drawings may include some equipment ident...,[]
8,toc1_e8,30,General arrangement drawings may provide some ...,[]
9,toc1_e9,30,Foundation layout may provide context for equi...,[]



=== TOC region 2 --  58 sections candidates ===
Sections retenues : ['toc2_e20', 'toc2_e21', 'toc2_e22', 'toc2_e23', 'toc2_e24', 'toc2_e25', 'toc2_e26', 'toc2_e27', 'toc2_e28', 'toc2_e29', 'toc2_e30', 'toc2_e31', 'toc2_e32', 'toc2_e49', 'toc2_e50', 'toc2_e54']


,section_id,relevance_score,reason,potential_information
0,toc2_e0,5,Safety instructions typically do not contain s...,[]
1,toc2_e1,10,Transport and packing sections usually focus o...,[]
2,toc2_e2,15,A packing list may include some identification...,[]
3,toc2_e3,15,Packing drawings may provide some context but ...,[]
4,toc2_e4,30,Installation instructions may contain some spe...,[]
5,toc2_e5,25,Lifting drawings may include dimensions but ar...,[]
6,toc2_e6,20,Preservation and storage procedures are unlike...,[]
7,toc2_e7,25,Skid positioning and leveling procedures may i...,[]
8,toc2_e8,20,Commissioning and startup sections are general...,[]
9,toc2_e9,25,Commissioning procedures may include some oper...,[]



=== TOC region 3 --  77 sections candidates ===
Sections retenues : ['toc3_e14', 'toc3_e16', 'toc3_e17', 'toc3_e71']


,section_id,relevance_score,reason,potential_information
0,toc3_e0,5,The introduction typically provides a general ...,[]
1,toc3_e1,20,The general description of the compressor may ...,[]
2,toc3_e2,30,The mechanism description may touch on technic...,[]
3,toc3_e3,25,The aspiration phase may include some operatio...,[]
4,toc3_e4,25,The compression phase may discuss operational ...,[]
5,toc3_e5,25,The discharge phase may include operational pa...,[]
6,toc3_e6,20,The definition of the volumetric ratio may be ...,[]
7,toc3_e7,20,Reasons for adjusting the volumetric ratio may...,[]
8,toc3_e8,20,The variable mechanism may discuss functionali...,[]
9,toc3_e9,20,External adjustment may relate to operational ...,[]



=== TOC region 4 --  12 sections candidates ===
Sections retenues : ['toc4_e1', 'toc4_e2', 'toc4_e4', 'toc4_e5', 'toc4_e6', 'toc4_e7']


,section_id,relevance_score,reason,potential_information
0,toc4_e0,60,The title suggests it may contain information ...,"[Manufacturer / Brand, Equipment name or type]"
1,toc4_e1,85,The title 'FAN DATA SHEET' indicates it likely...,"[Model / Type / Reference, Technical specifica..."
2,toc4_e2,70,The title suggests it may contain operational ...,"[Technical specifications, Operating parameter..."
3,toc4_e3,30,The title is vague and does not suggest specif...,[]
4,toc4_e4,60,The title indicates it may contain specificati...,"[Technical specifications, Dimensions, Weight]"
5,toc4_e5,50,The title suggests it may contain process data...,"[Operating parameters, Technical specifications]"
6,toc4_e6,60,"Similar to section 0, the title suggests it ma...","[Manufacturer / Brand, Equipment name or type]"
7,toc4_e7,85,The title 'Fa Da Se' likely refers to fan data...,"[Model / Type / Reference, Technical specifica..."
8,toc4_e8,20,The title is unclear and does not suggest any ...,[]
9,toc4_e9,10,The title is vague and does not suggest any sp...,[]



=== TOC region 5 --  31 sections candidates ===
Sections retenues : []


,section_id,relevance_score,reason,potential_information
0,toc5_e0,10,The introduction typically provides a general ...,[]
1,toc5_e1,30,General installation and operation instruction...,[]
2,toc5_e2,20,Storage instructions are unlikely to contain s...,[]
3,toc5_e3,20,Characteristics for storage areas may not incl...,[]
4,toc5_e4,20,This section is focused on storage procedures ...,[]
5,toc5_e5,20,Inspection procedures may touch on equipment b...,[]
6,toc5_e6,20,This section discusses storage during pre-comm...,[]
7,toc5_e7,20,Storage during equipment downtime is unlikely ...,[]
8,toc5_e8,20,This section discusses components for reassemb...,[]
9,toc5_e9,30,Transport and lifting instructions may contain...,[]


## 6. Vue d'ensemble : sections retenues, toutes régions confondues


In [29]:
rows = []
for i, (entries, result) in enumerate(zip(organized_regions, results_by_region), 1):
    if result is None:
        continue
    selected_ids = set(result.selected_sections)
    for j, entry in enumerate(entries):
        section_id = f"toc{i}_e{j}"
        rows.append({
            "toc_region": i,
            "section_id": section_id,
            "title": entry.text,
            "page": entry.printed_page_ref,
            "retenue": section_id in selected_ids,
        })

if rows:
    overview = pd.DataFrame(rows)
    display(overview.head(10))
    print("Total retenues :", int(overview["retenue"].sum()), "/", len(overview))
else:
    print("Aucun resultat LLM disponible (verifiez OPENAI_API_KEY dans .env).")


,toc_region,section_id,title,page,retenue
0,1,toc1_e0,DOCUMENTS&DRAWINGSLIST,P1-REF-LST-12-123-007,False
1,1,toc1_e1,SUBVENDORLIST,P1-REF-LST-12-123-003,False
2,1,toc1_e2,NOISEDATASHEET,P1-REF-DSS-12-123-008,False
3,1,toc1_e3,LUBRICANTLIST,P1-REF-SPC-12-123-002,False
4,1,toc1_e4,PAINTINGSPECIFICATIONS,P1-REF-SPC-12-123-003,False
5,1,toc1_e5,PACKAGEDRAWINGS,NaN,False
6,1,toc1_e6,PROCESSFLOWDIAGRAM,P1-REF-2012-123-014,False
7,1,toc1_e7,P&IDREFRIGERATIONUNIT,P1-REF-2012-123-001,False
8,1,toc1_e8,GENERALARRANGEMENTDRAWING,P1-REF-2012-123-003,False
9,1,toc1_e9,FOUNDATIONLAYOUT,P1-REF-2012-123-012,False


Total retenues : 42 / 296


In [31]:
retenues = overview[overview["retenue"]].reset_index(drop=True)
print(f"\n=== Sections retenues (total {len(retenues)}) ===")
display(retenues.head(10))



=== Sections retenues (total 42) ===


,toc_region,section_id,title,page,retenue
0,1,toc1_e11,AIRCOOLERCONDENSERDATASHEET,P1-REF-DSS-12-123-011,True
1,1,toc1_e12,AIRCOOLERCONDENSERGENERALARRANGEMENT,P1-REF-2012-123-005,True
2,1,toc1_e17,ECONOMIZERDATASHEET,P1-REF-DSS-12-123-012,True
3,1,toc1_e19,KODRUMDATASHEET,P1-REF-DSS-12-123-013,True
4,1,toc1_e21,OILCOOLERDATASHEET,P1-REF-DSS-12-123-014,True
5,1,toc1_e23,RECEIVERDATASHEET,P1-REF-DSS-12-123-017,True
6,1,toc1_e25,OILSEPARATORDATASHEET,P1-REF-DSS-12-123-016,True
7,1,toc1_e27,SUCTIONFILTERDATASHEET,P1-REF-DSS-12-123-018,True
8,1,toc1_e29,OILFILTERDATASHEET,P1-REF-DSS-12-123-015,True
9,1,toc1_e39,COMPRESSORDATASHEETS,P1-REF-DSS-12-123-003,True


## 7. Pour aller plus loin

- `ocr_table_entries_for_page` réutilise `TOCDetector._parse_table` tel
  quel -- si le format des lignes OCR diffère trop du format natif
  (colonnes fusionnées, bruit), c'est ce parseur qu'il faut ajuster, pas
  cette cellule.
- `organize_region` dédoublonne sur le texte du titre en minuscules ; sur
  un vrai document, affinez si les titres OCR et natifs diffèrent
  légèrement (accents, casse, espaces).
- Le nombre d'appels LLM #1 ici est **un par région de TOC détectée**
  (pas un par page, pas un global) -- ajustez `region_entries_to_payload`
  si vous préférez un seul appel fusionnant toutes les régions.
